# Notebook 2: Creating Agents Properly

**What you'll learn:** Agent instructions design, output types, model settings, structured output, and real agentic patterns — WITHOUT handoffs or guardrails.

---

## The Agentic Workflow Mindset

Traditional AI:
```
User → Prompt → LLM → Response → Done
```

Agentic AI (2025-2026 paradigm):
```
User → Agent → [Think → Act → Observe → Think → Act → ...] → Result
```

The key difference: **the agent decides HOW to complete the task, not just WHAT to say.**

In [1]:
# uv add openai-agents pydantic

## Local Model Setup (Ollama + llama3.2:latest)

In [2]:
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, ModelSettings
from agents.tracing import set_tracing_disabled

# Disable tracing — not needed for local models
set_tracing_disabled(True)

# Ollama client
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# Local model
local_model = OpenAIChatCompletionsModel(
    model="llama3.2:latest",
    openai_client=ollama_client
)

print("Local model ready: llama3.2:latest")

Local model ready: llama3.2:latest


---

## Agent Anatomy — Every Parameter Explained

```python
Agent(
    name="...",              # Identity (used in tracing/debugging)
    instructions="...",      # System prompt — the agent's brain
    model=local_model,       # Which LLM to use (local Ollama)
    tools=[...],             # Functions the agent can call
    output_type=MyModel,     # Force structured JSON output (Pydantic)
    handoffs=[...],          # Other agents it can delegate to (Notebook 4+)
    guardrails=[...],        # Input/output validation (Notebook 5+)
    model_settings=...,      # Temperature, top_p, max_tokens, etc.
)
```

---

## Instructions Design — The Most Important Skill

The `instructions` field is your agent's **system prompt**. It defines:
- WHO the agent is (role/persona)
- WHAT it should do (task)
- HOW it should behave (constraints/rules)
- WHEN to use tools (decision criteria)

In [3]:
# BAD instructions — too vague
bad_agent = Agent(
    name="Bad Agent",
    instructions="You are helpful.",
    model=local_model
)

# GOOD instructions — specific, structured, actionable
good_agent = Agent(
    name="Code Reviewer",
    instructions="""
    You are a senior Python code reviewer at a tech company.
    
    YOUR ROLE:
    - Review Python code for bugs, style issues, and performance problems.
    - Rate code quality from 1-10.
    - Suggest specific improvements with corrected code snippets.
    
    YOUR RULES:
    - Always check for: type hints, docstrings, error handling, edge cases.
    - Be constructive, not harsh.
    - If the code is good, say so — don't invent problems.
    - Keep reviews concise (under 200 words).
    
    OUTPUT FORMAT:
    Score: X/10
    Issues: [list]
    Suggestions: [list with code]
    """,
    model=local_model
)

result = await Runner.run(good_agent, """
Review this code:

def add(x, y):
    return x + y
""")

print(result.final_output)

Score: 6/10

Issues:
- No docstring is provided for the function. A brief description of what it does should be included.
- No error handling is in place for cases where either `x` or `y` are not numbers.

Suggestions:
```python
def add(x, y) -> float:
    """
    Returns the sum of two numbers.
    
    Args:
        x (float): The first number.
        y (float): The second number.
    
    Raises:
        TypeError: If either x or y is not a number.
    
    Returns:
        float: The sum of x and y.
    """
    if not isinstance(x, (int, float)) or not isinstance(y, (int, float)):
        raise TypeError("Both inputs must be numbers.")
    return x + y
```
I've added type hints for the function's return value and its parameters, which makes it clearer what types are expected. The function now raises a `TypeError` if either input is not a number, preventing potential bugs that might occur due to non-numeric values being passed in.


---

## Model Settings — Control LLM Behavior

In [4]:
# Creative agent — high temperature
creative_agent = Agent(
    name="Poet",
    instructions="You write creative, imaginative descriptions. Be vivid and poetic.",
    model=local_model,
    model_settings=ModelSettings(
        temperature=0.9,       # High = more creative/random
        max_tokens=500,        # Limit output length
    )
)

# Precise agent — low temperature
precise_agent = Agent(
    name="Fact Checker",
    instructions="You provide precise, factual answers. No speculation. Cite sources if possible.",
    model=local_model,
    model_settings=ModelSettings(
        temperature=0.1,       # Low = more deterministic/precise
        max_tokens=300,
    )
)

# Same question, different agents
question = "Describe the Badshahi Mosque in Lahore."

creative_result = await Runner.run(creative_agent, question)
precise_result = await Runner.run(precise_agent, question)

print(" CREATIVE:\n", creative_result.final_output[:300])
print("\n PRECISE:\n", precise_result.final_output[:300])

 CREATIVE:
 Amidst the storied city of Lahore, where Mughal splendor meets the passage of time, stands the majestic Badshahi Mosque. This architectural marvel, a testament to the grandeur of the Mughal Empire, is an awe-inspiring fusion of Islamic and Persianate design elements.

The mosque's imposing façade ri

 PRECISE:
 The Badshahi Mosque (Urdu: بادشاہی مسجد) is a Mughal-era mosque located in Lahore, Pakistan. It was built during the reign of Emperor Aurangzeb (1658-1707) and completed in 1673.

The mosque is considered one of the largest and most impressive examples of Mughal architecture in India and Pakistan. I


---

## Structured Output — Force JSON Responses with Pydantic

This is a **game-changer** for production agents. Instead of parsing messy text, get clean typed data.

**Note:** With local models, add explicit JSON instructions to help model output valid JSON.

In [5]:
from pydantic import BaseModel, Field


# Define the output structure
class PodcastReview(BaseModel):
    title: str = Field(description="Podcast title")
    host: str = Field(description="Podcast host(s)")
    rating: float = Field(description="Rating from 1.0 to 10.0")
    genre: str = Field(description="Primary genre (e.g., AI Research, Machine Learning, AI News, AI Ethics)")
    technical_level: str = Field(description="Beginner, Intermediate, or Advanced")
    summary: str = Field(description="One-sentence summary")
    recommendation: bool = Field(description="Would you recommend this?")


# Create agent with structured output
reviewer = Agent(
    name="AI Podcast Reviewer",
    instructions=(
        "You are an AI podcast critic. Analyze podcasts about artificial intelligence, "
        "machine learning, and related topics. Provide structured reviews and assess "
        "the technical depth and accessibility of each podcast. "
        "IMPORTANT: You must respond with ONLY valid JSON, no other text."
    ),
    model=local_model,
    output_type=PodcastReview,
)

result = await Runner.run(reviewer, "Review the podcast 'Lex Fridman Podcast'")

# result.final_output is now a PodcastReview object, not a string!
review = result.final_output
print(f"Title:          {review.title}")
print(f"Host:           {review.host}")
print(f"Rating:         {review.rating}/10")
print(f"Genre:          {review.genre}")
print(f"Tech Level:     {review.technical_level}")
print(f"Summary:        {review.summary}")
print(f"Recommended:    {'Yes' if review.recommendation else 'No'}")

Title:          Lex Fridman Podcast
Host:           Lex Fridman
Rating:         4.8/10
Genre:          Science, Technology, Philosophy, Conversation
Tech Level:     Advanced
Summary:        In-depth discussions on AI, ML, philosophy, and various conversations with experts.
Recommended:    Yes


### Why Structured Output Matters for Agentic Workflows:

```
Without output_type:  "The movie is great, I'd give it 8/10..." → Parse with regex? 
With output_type:     MovieReview(rating=8.0, ...)              → Use directly! 
```

In multi-agent systems, structured output lets one agent's output become another agent's input **reliably**.

---

## Real-World Pattern: E-Commerce Product Classifier Agent

**Scenario:** You run an online store. Customers describe products they want. Your agent classifies and routes them.

In [6]:
from pydantic import BaseModel, Field
from typing import Literal


class ProductClassification(BaseModel):
    category: Literal["electronics", "clothing", "food", "books", "other"] = Field(
        description="Product category"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgently does the customer need this?"
    )
    price_range: Literal["budget", "mid", "premium"] = Field(
        description="Estimated price range based on description"
    )
    search_query: str = Field(
        description="Optimized search query for the product catalog"
    )
    confidence: float = Field(
        description="Confidence score 0.0-1.0"
    )


classifier = Agent(
    name="Product Classifier",
    instructions="""
    You classify customer product requests for an e-commerce platform.
    Analyze the customer's natural language description and extract:
    - What category the product falls into
    - How urgent the request seems
    - Estimated price range
    - An optimized search query
    - Your confidence in the classification
    
    Be accurate. If you're unsure, set confidence low.
    IMPORTANT: You must respond with ONLY valid JSON, no other text.
    """,
    model=local_model,
    output_type=ProductClassification,
    model_settings=ModelSettings(temperature=0.1)
)

# Test with different customer requests
requests = [
    "I need a laptop for my son's school, nothing too expensive",
    "Looking for a birthday gift, maybe a nice novel",
    "URGENT: Need a phone charger, mine just broke!",
]

for req in requests:
    result = await Runner.run(classifier, req)
    c = result.final_output
    print(f"\nRequest: '{req}'")
    print(f"   Category: {c.category} | Urgency: {c.urgency} | Price: {c.price_range}")
    print(f"   Search: '{c.search_query}' | Confidence: {c.confidence:.0%}")


Request: 'I need a laptop for my son's school, nothing too expensive'
   Category: electronics | Urgency: low | Price: budget
   Search: 'affordable laptops for students' | Confidence: 80%

Request: 'Looking for a birthday gift, maybe a nice novel'
   Category: books | Urgency: low | Price: budget
   Search: 'birthday gift novels' | Confidence: 80%

Request: 'URGENT: Need a phone charger, mine just broke!'
   Category: electronics | Urgency: high | Price: budget
   Search: 'phone chargers' | Confidence: 90%


---

## Dynamic Instructions — Instructions as Functions

Instructions can be a **function** that returns a string, letting you inject runtime context:

In [7]:
from datetime import datetime


def dynamic_instructions(context, agent):
    """Instructions that change based on current state."""
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M")
    return f"""
    You are a scheduling assistant.
    Current date/time: {current_time}
    
    Rules:
    - If it's morning (before 12pm), suggest productive tasks.
    - If it's afternoon, suggest meetings and collaborations.
    - If it's evening, suggest winding down activities.
    - Always be aware of the current time in your suggestions.
    """


scheduler = Agent(
    name="Smart Scheduler",
    instructions=dynamic_instructions,  # <-- Pass function, not string!
    model=local_model
)

result = await Runner.run(scheduler, "What should I do right now?")
print(result.final_output)

Currently the time is 18:20 on April 10th, 2026. Since it's in the late afternoon/early evening, I would suggest winding down activities to help you relax after a productive day.

Why not try some gentle stretches or yoga poses to release any tension and calm your mind? Alternatively, you could take a warm bath or listen to soothing music to unwind and prepare for the rest of your evening.


---

## Real-World Pattern: Customer Support Ticket Analyzer

**Industry Problem:** Thousands of support tickets daily. Agent analyzes sentiment, priority, and routes them.

In [8]:
from pydantic import BaseModel, Field
from typing import Literal


class TicketAnalysis(BaseModel):
    sentiment: Literal["positive", "neutral", "negative", "angry"] = Field(
        description="Customer sentiment"
    )
    priority: Literal["P0-critical", "P1-high", "P2-medium", "P3-low"] = Field(
        description="Ticket priority"
    )
    department: Literal["billing", "technical", "sales", "general"] = Field(
        description="Which department should handle this"
    )
    summary: str = Field(description="One-line summary for the dashboard")
    suggested_response: str = Field(description="Draft first response to customer")


ticket_analyzer = Agent(
    name="Ticket Analyzer",
    instructions="""
    You are an AI support ticket analyzer for a SaaS company.
    
    PRIORITY RULES:
    - P0-critical: Service down, data loss, security breach
    - P1-high: Feature broken, payment issues, angry customer
    - P2-medium: Bug reports, feature requests, how-to questions
    - P3-low: General feedback, suggestions, non-urgent inquiries
    
    DEPARTMENT ROUTING:
    - billing: Payment, subscription, invoice, refund
    - technical: Bugs, errors, API issues, integrations
    - sales: Pricing, enterprise plans, demos, upgrades
    - general: Everything else
    
    Write the suggested_response in a professional, empathetic tone.
    IMPORTANT: You must respond with ONLY valid JSON, no other text.
    """,
    model=local_model,
    output_type=TicketAnalysis,
    model_settings=ModelSettings(temperature=0.2)
)

# Simulate incoming tickets
tickets = [
    "Your API has been returning 500 errors for 2 hours. Our production is DOWN. Fix this NOW!",
    "Hi, I was charged twice this month. Can you please look into it? Thanks.",
    "Love the product! Any chance you'll add dark mode soon?",
]

for ticket in tickets:
    result = await Runner.run(ticket_analyzer, ticket)
    t = result.final_output
    print(f"\n{'='*60}")
    print(f"Ticket: {ticket[:60]}...")
    print(f"Sentiment: {t.sentiment}")
    print(f"Priority:  {t.priority}")
    print(f"Route to:  {t.department}")
    print(f"Summary:   {t.summary}")
    print(f"Response:  {t.suggested_response[:150]}...")


Ticket: Your API has been returning 500 errors for 2 hours. Our prod...
Sentiment: angry
Priority:  P0-critical
Route to:  technical
Summary:   Production down due to API error, urgent attention required.
Response:  I apologize for the inconvenience. Our team is actively working on resolving the issue. Can you please provide more information about the error messag...

Ticket: Hi, I was charged twice this month. Can you please look into...
Sentiment: neutral
Priority:  P3-low
Route to:  billing
Summary:   We apologize for the inconvenience. We will investigate the double charge and resolve the issue as soon as possible.
Response:  {"response": "We apologize for the inconvenience. We will investigate the double charge and resolve the issue as soon as possible.", "reference_number...

Ticket: Love the product! Any chance you'll add dark mode soon?...
Sentiment: positive
Priority:  P3-low
Route to:  general
Summary:   Thank you for your suggestion. We appreciate your feedback and will con

---

## Agent Design Best Practices (2025-2026)

| Practice | Why |
|---|---|
| **Single Responsibility** | Each agent does ONE thing well |
| **Specific Instructions** | Vague = unpredictable, Specific = reliable |
| **Use output_type** | Structured data > parsing text |
| **Low temperature for tasks** | Deterministic results for business logic |
| **Dynamic instructions** | Inject runtime context (time, user data, state) |
| **Test with edge cases** | Empty input, adversarial input, long input |

---

## Summary

| Concept | Code |
|---|---|
| Model settings | `ModelSettings(temperature=0.1, max_tokens=500)` |
| Structured output | `Agent(output_type=MyPydanticModel)` |
| Dynamic instructions | `Agent(instructions=my_function)` |
| Production pattern | Classify → Route → Respond |

**Next:** Notebook 3 — Running all of this locally with Ollama (zero cost, full privacy).